<a href="https://colab.research.google.com/github/mikysetiawan/MachineLearningTerapan/blob/master/Tugas2_RecommendationArticle.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import kagglehub
from kagglehub import KaggleDatasetAdapter

import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [2]:
df = kagglehub.dataset_load(
    KaggleDatasetAdapter.PANDAS,
    "viniciuslambert/medium-data-science-articles-dataset",
    "medium-data-science-articles-2020.csv"
)

print("First 5 records:", df.head())

100%|██████████| 9.10M/9.10M [00:00<00:00, 67.2MB/s]

Extracting zip of medium-data-science-articles-2020.csv...


First 5 records:                                                  url  \
0  https://towardsdatascience.com/making-python-p...   
1  https://towardsdatascience.com/how-to-be-fancy...   
2  https://uxdesign.cc/how-exactly-do-you-find-in...   
3  https://towardsdatascience.com/from-scratch-to...   
4  https://www.cantorsparadise.com/the-waiting-pa...   

                                               title            author  \
0              Making Python Programs Blazingly Fast      martin.heinz   
1                        How to be fancy with Python           dipam44   
2  How exactly do you find insights from qualitat...   taylornguyen144   
3  From scratch to search: playing with your data...  stanislavprihoda   
4  The Waiting Paradox: An Intro to Probability D...        maikeelisa   

                                        author_page  \
0      https://towardsdatascience.com/@martin.heinz   
1           https://towardsdatascience.com/@dipam44   
2              https://uxdesign.cc/@

In [3]:
df.info()

print('Jumlah data artikel: ', len(df.url.unique()))
print('Jumlah data author: ', len(df.author.unique()))
print('Jumlah data tag: ', len(df.tag.unique()))

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 108021 entries, 0 to 108020
Data columns (total 10 columns):
 #   Column        Non-Null Count   Dtype  
---  ------        --------------   -----  
 0   url           108021 non-null  object 
 1   title         108021 non-null  object 
 2   author        108021 non-null  object 
 3   author_page   108021 non-null  object 
 4   subtitle      38586 non-null   object 
 5   claps         108021 non-null  float64
 6   responses     108021 non-null  int64  
 7   reading_time  108021 non-null  int64  
 8   tag           108021 non-null  object 
 9   date          108021 non-null  object 
dtypes: float64(1), int64(2), object(7)
memory usage: 8.2+ MB
Jumlah data artikel:  107971
Jumlah data author:  46809
Jumlah data tag:  7


In [4]:
# Mengecek missing value pada dataframe
df.isnull().sum()

,0
url,0
title,0
author,0
author_page,0
subtitle,69435
claps,0
responses,0
reading_time,0
tag,0
date,0


In [5]:
# Mengecek tag artikel yang unik
df.tag.unique()

array(['Data Science', 'Machine Learning', 'Artificial Inteligence',
       'Deep Learning', 'Data', 'Big Data', 'Analytics'], dtype=object)

In [6]:
# Pembersihan Data
df_clean = df.drop_duplicates(subset='title').reset_index(drop=True)

In [7]:
# Penggabungan Fitur (Title + Tag) agar lebih akurat
# Mengisi subtitle yang kosong dengan string kosong
df_clean['subtitle'] = df_clean['subtitle'].fillna('')
df_clean['content_features'] = df_clean['title'] + " " + df_clean['tag']

# Model Development

In [13]:
# Inisialisasi TfidfVectorizer
tf = TfidfVectorizer(stop_words='english')

# Melakukan perhitungan idf pada data tag
tf.fit(df_clean['content_features'])

# Mapping array dari fitur index integer ke fitur nama
tf.get_feature_names_out()

array(['00', '000', '0001', ..., '𝗲𝘃𝗼𝗹𝘂𝘁𝗶𝗼𝗻', '𝗶𝗻𝘁𝗲𝗿𝗽𝗿𝗲𝘁𝗮𝘁𝗶𝗼𝗻', '𝗼𝗳'],
      dtype=object)

In [18]:
# Melakukan fit lalu ditransformasikan ke bentuk matrix
tfidf_matrix = tf.fit_transform(df_clean['content_features'])

# Melihat ukuran matrix tfidf
tfidf_matrix.shape

(105553, 48885)

In [25]:
def get_recommendations(target_title, k=5):
    # 1. Cari index artikel berdasarkan judul (Gunakan df_clean agar index sinkron dengan matrix)
    try:
        # Cari yang mengandung kata tersebut (lebih fleksibel)
        idx = df_clean[df_clean['title'].str.contains(target_title, case=False)].index[0]
        actual_title = df_clean.iloc[idx]['title']
    except IndexError:
        return "Artikel tidak ditemukan. Coba gunakan kata kunci lain."

    # 2. Hitung similarity HANYA untuk artikel ini terhadap seluruh dataset
    query_vector = tfidf_matrix[idx]
    cosine_sim_scores = cosine_similarity(query_vector, tfidf_matrix).flatten()

    # 3. Ambil index k+1 skor tertinggi (karena index 0 biasanya artikel itu sendiri)
    related_indices = cosine_sim_scores.argsort()[-(k+1):][::-1]

    # 4. Ambil data asli untuk ditampilkan
    results = []
    for i in related_indices:
        if i != idx: # Jangan masukkan artikel yang sedang dicari
            results.append({
                'title': df_clean.iloc[i]['title'],
                'tag': df_clean.iloc[i]['tag'],
                'score': cosine_sim_scores[i]
            })

    print(f"Hasil rekomendasi untuk artikel: {actual_title}")
    return pd.DataFrame(results).head(k)

In [28]:
df_clean[df_clean.title.eq('Making Python Programs Blazingly Fast')]

,url,title,author,author_page,subtitle,claps,responses,reading_time,tag,date,content_features
0,https://towardsdatascience.com/making-python-p...,Making Python Programs Blazingly Fast,martin.heinz,https://towardsdatascience.com/@martin.heinz,Let’s look at the performance of our Python pr...,3300.0,3,5,Data Science,2020-01-01,Making Python Programs Blazingly Fast Data Sci...


In [29]:
# Mendapatkan rekomendasi artikel
get_recommendations('Making Python Programs Blazingly Fast')

Hasil rekomendasi untuk artikel: Making Python Programs Blazingly Fast


,title,tag,score
0,What’s missing from all Data Science training ...,Data Science,0.352522
1,Making Data F.A.I.R.,Data,0.349723
2,"How to Learn R for Data Science, Fast!",Data Science,0.329278
3,Making Data Trees in Python,Data Science,0.314531
4,Facebook’s Artificial Intelligence Programs,Artificial Inteligence,0.313164
